# OpenAI API Basics with Real World Applications

**Author:** Sebastian Galindo

This is my entry and experience experimenting with OpenAI's api and testing it for what I think could be some real world applications of it right now, at least to me. My goal with this notebook is to guide you through my thought process and highlight things I consider important and worth noting.

First, as what is usual with jupyter notebooks, your imports. In this case we are using the following imports:

* **os and dotenv:** used to load the .env config file to load your OPENAI api key.
* **website_scraper:** is a local call to an script called "website_scraper.py" located at "ai-lab\llm-engineering\openai-api-basics".
* **IPython.display:** a library used to display in Markdown the result of our api call.
* **openai:** the heart of this script, the library that allows us to call OPENAI's api.

In [7]:
# imports

import os
from dotenv import load_dotenv
from website_scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

First thing we do is load our OpenAI api key from the .env file. Added some minor validation to check for the key.

In [8]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


Next, we need to define the message we are going to send to OpenAI's api. We define a "message" variable and add it to a "messages" list. It is important to define the role for that message. More on what those roles are for and what they are in a minute.

In [9]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

Next, we can make OpenAI's api a request with the following code. Notice that in the parameter "model" for the create() function we specify which OpenAI LLM model we want the api to use to respond to our message. By default the api returns 1 response in the "choices[]" array. To change how many responses you get you can specify a parameter "n" in the create() function as follow:


    response = openai.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
        n=3
    )

In [10]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'Hi there! Welcome, and nice to meet you. Thanks for saying hi.\n\nI’m here to help with a wide range of things. Some ideas if you’re not sure where to start:\n- Explain concepts or answer questions (anything from science to history to everyday stuff)\n- Help with writing, editing, or brainstorming (emails, essays, stories, captions)\n- Solve math or programming problems\n- Plan or organize (study schedules, trip plans, projects)\n- Translate or practice a new language\n- Learn something new with quick explanations or mini-lessons\n- Have a friendly chat, trivia, or jokes\n\nTell me what you’re into or what you’d like to do today. Do you want a quick ice-breaker like a fun fact or a mini riddle? Or describe a task and I’ll dive in.'

Just a quick look of how an OpenAI's api response looks like

In [11]:
print(response)

ChatCompletion(id='chatcmpl-DmsEUEnAGt9vQMRVw4DAHAgT7moZs', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hi there! Welcome, and nice to meet you. Thanks for saying hi.\n\nI’m here to help with a wide range of things. Some ideas if you’re not sure where to start:\n- Explain concepts or answer questions (anything from science to history to everyday stuff)\n- Help with writing, editing, or brainstorming (emails, essays, stories, captions)\n- Solve math or programming problems\n- Plan or organize (study schedules, trip plans, projects)\n- Translate or practice a new language\n- Learn something new with quick explanations or mini-lessons\n- Have a friendly chat, trivia, or jokes\n\nTell me what you’re into or what you’d like to do today. Do you want a quick ice-breaker like a fun fact or a mini riddle? Or describe a task and I’ll dive in.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

Okay, so far we have created a request to OpenAI's api and got a response, good. 

But now, what could we do with it? Here's a simple example of a function that scrapes the content of a server side webpage. To do this we use the fetch_website_content() function from the website-scraper.py script.

In [12]:
# Let's try out this utility

website = fetch_website_contents("https://cnn.com")
print(website)

Breaking News, Latest News and Videos | CNN

CNN values your feedback
1. How relevant is this ad to you?
2. Did you encounter any technical issues?
Video player was slow to load content
Video content never loaded
Ad froze or did not finish loading
Video content did not start after ad
Audio on ad was too loud
Other issues
Ad never loaded
Ad prevented/slowed the page from loading
Content moved around while ad loaded
Ad was repetitive to ads I've seen previously
Other issues
Cancel
Submit
Thank You!
Your effort and contribution in providing this feedback is much
                                        appreciated.
Close
Ad Feedback
Close icon
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
World Cup 2026
Ukraine-Russia War
Israel-Hamas War
Games
More
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
World Cup 2026
Ukraine-Russia War
Israel-Hamas War
Games
Watch
Listen
Live TV
Subscribe
Sign in
My Account

Now, I told you a second ago that I was going to explain what the "role" was on the message object sent to OpenAI's api, so here it is.

There are 3 possible roles in the message object and each one of them has a functionality or scenario where you would want to use them.

* **system:** The system role is used to set to the model the context, set the model's behavior, it's persona or rules before the conversation with the model starts.
* **user:** This role is the one you want to use when you actually want to ask the model to respond to some request. This would be the actual input or question.
* **assistant:** This role is used when you want to include previous model responses as part of the conversation history. Without this last role each conversation starts fresh without the model "remembering" what it respond before.

In [19]:
# Define our system prompt, add some context, rules, persona, what ever you want to customize the model's response.

system_prompt = """
You are a helpful assistant that analyzes the contents of a website,
and provides a short and precise summary in Spanish, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

Now we define the user prompt, you know, the actual input or question we asking the model.

In [14]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, don't summarize these too.

"""

In [16]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_prefix + website}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

'Este sitio web de CNN ofrece noticias actuales, videos y contenido multimedia sobre una variedad de temas como política, negocios, salud, entretenimiento, estilo de vida, viajes, deportes, ciencia y clima. Incluye también coberturas especiales de eventos internacionales importantes como la Guerra en Ucrania-Rusia, el conflicto entre Israel y Hamas, y la Copa del Mundo 2026. Además, proporciona secciones para escuchar noticias en vivo y acceder a diferentes ediciones regionales en varios idiomas.'

For simplicity we can define a simple function that creates the same structure as above just passing the website we want the model to analyze as a parameter.

In [17]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [20]:

messages_for(website)

[{'role': 'system',
  'content': '\nYou are a helpful assistant that analyzes the contents of a website,\nand provides a short and precise summary in Spanish, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': "\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, don't summarize these too.\n\nBreaking News, Latest News and Videos | CNN\n\nCNN values your feedback\n1. How relevant is this ad to you?\n2. Did you encounter any technical issues?\nVideo player was slow to load content\nVideo content never loaded\nAd froze or did not finish loading\nVideo content did not start after ad\nAudio on ad was too loud\nOther issues\nAd never loaded\nAd prevented/slowed the page from loading\nContent moved around while ad loaded\nAd was repetitive to ads I've seen previously\nOther issues\nCancel\nSubmit\nT

Now, we can create a little "summarize()" function that receives the URL of a website and gives a response using the messages_for() function created above!

In [21]:
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [22]:
summarize("https://cnn.com")

'CNN es un sitio web de noticias que ofrece información actualizada en diversas categorías como Estados Unidos, mundo, política, negocios, salud, entretenimiento, estilo, viajes, deportes, ciencia, clima y más. Además, proporciona contenido en varios idiomas, incluyendo inglés, árabe y español. El sitio incluye acceso a televisión en vivo, videos, podcasts y newsletters, permitiendo a los usuarios personalizar su experiencia siguiendo temas de interés y gestionando sus cuentas. También cuenta con secciones especiales sobre eventos y conflictos internacionales actuales como la guerra Ucrania-Rusia y el Mundial 2026.'

As is, the model output comes as a large string with some markdown format, we can create a function to parse this string and give it a proper Markdown format.

In [23]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [24]:
display_summary("https://cnn.com")

CNN es un sitio web de noticias que ofrece cobertura global en diversas categorías como política, negocios, salud, entretenimiento, deportes, ciencia, clima y más. Proporciona contenido en varios idiomas, incluyendo inglés, árabe y español. Además de noticias escritas, el sitio incluye videos, transmisiones en vivo y podcasts. También permite a los usuarios personalizar su experiencia mediante cuentas personales y seguimiento de temas específicos.

here's another test but with Anthropics website.

In [25]:
display_summary("https://anthropic.com")

Anthropic es una empresa pública dedicada a la investigación y desarrollo de inteligencia artificial, enfocada en la seguridad y el beneficio social. Ofrecen productos basados en IA, como Claude y sus variantes (Claude Code, Claude Cowork, etc.), destinados a uso profesional y técnico. La empresa promueve la transparencia, políticas responsables y cumplimiento en seguridad. Además, Anthropic proporciona recursos educativos, documentación para desarrolladores y una plataforma para interactuar con sus modelos de IA. Su misión es maximizar los beneficios de la IA minimizando sus riesgos.

To make things a bit more dinamic we can create a special function to not only create the request to OpenAI's api by giving it an URL, but also specifying the model we want to use in a new parameter!

In [31]:
# Try a different model and compare outputs
def summarize_with_model(url, model):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model=model,
        messages=messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
# A function to display this nicely in the output, using markdown. 
# Similar to the previous one but adding the model parameter

def display_summary_with_model(url, model):
    summary = summarize_with_model(url, model)
    display(Markdown(summary))

Now let's test how to different OpenAI models responds!

In [33]:
print(display_summary_with_model("https://anthropic.com", "gpt-4.1-nano"))

Este sitio web pertenece a Anthropic, una organización dedicada a la investigación y desarrollo de inteligencia artificial segura y responsable. Ofrecen productos como Claude y sus variantes, así como recursos educativos, políticas de transparencia y objetivos de responsabilidad. También destacan su enfoque en la seguridad, la innovación y el impacto positivo de la IA en la sociedad.

None


In [34]:
print(display_summary_with_model("https://anthropic.com", "gpt-4.1-mini"))

Anthropic es una empresa dedicada a la investigación y desarrollo de inteligencia artificial con un enfoque prioritario en la seguridad y la mitigación de riesgos asociados. Ofrece diversos productos basados en IA como Claude, Claude Code, Claude Cowork y Claude Security, además de plataformas y modelos avanzados (Opus, Sonnet, Haiku). La organización también cuenta con recursos educativos como Anthropic Academy, tutoriales y documentación para desarrolladores. Su misión es asegurar que los beneficios de la IA sean accesibles y gestionados responsablemente.

None


This is awesome! The full power of OpenAI LLM models right available in your python code!
But seriously... what useful real world applications could there be for some code like this?

Here are a couple examples I can think of, based of current needs and interests.

## Study Case: League of Legends Patch Notes Content Generator

If you know me you would know I'm a big League of Legends fan (yes, the videogame). But what many people don't know is that I create content for it too! (link of my twitch in my bio).

So thinking about one real world application for this type of code would be the following. Since I create content for LoL, one of the pieces of content I make is a "Patch Notes Post" where I post what changes are happening in a new patch note version. So what I could ask an AI model to give me the script to explain in a short form content video what the big updates are and what characters is the game updating? Let's give it a try.

So first, let me define the messages structure of what are we sending to the model to accomplish what I just said.

In [42]:
# Define our system prompt, add some context, rules, persona, what ever you want to customize the model's response.

system_prompt = """
You are a content creator for League of Legends that streams on twitch and creates short-form content for platforms like tiktok and instagram.
You provide concise and clear summaries for patch notes of LoL in the form of a script for a video.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [43]:
# Define our user prompt

user_prompt_prefix = """
Here are the patch notes .
Provide a short summary of the patch notes.
Give me a list of champions buffed and champions nerfed if available.

"""

Let's now add some URLs of League patch notes and generate a summary of each one of them.

In [44]:
leaguenotes = [
    "https://www.leagueoflegends.com/en-us/news/game-updates/league-of-legends-patch-26-9-notes/",
    "https://www.leagueoflegends.com/en-us/news/game-updates/league-of-legends-patch-26-10-notes/",
    "https://www.leagueoflegends.com/en-us/news/game-updates/league-of-legends-patch-26-11-notes/"
]

In [45]:
for url in leaguenotes:
    print(f"\n--- {url} ---")
    display_summary(url)


--- https://www.leagueoflegends.com/en-us/news/game-updates/league-of-legends-patch-26-9-notes/ ---


# League of Legends Patch 26.9 Summary

Patch 26.9 brings a demon-hunting theme with new skins and updates across champions, items, systems, and runes. Key highlights include:

- **Rune Changes:** Deathfire Touch makes a comeback with a slightly nerfed damage scaling to balance its power.
- **Role Quests:** Simplified progression and reduced penalties to improve player experience, especially towards the end of laning phase.
- **Champion Updates:** Shyvana receives buffs enhancing her AD and AP builds, adding more durability for Dragon-themed paths.
- **Gameplay:** New starting items and alternate champion builds to experiment with.
- **ARAM & Arena:** More bug fixes, an update to ARAM: Mayhem, a new map for the Arena, and WASD functionality implemented in Ranked.
- **Skins:** Launch of Pandemonium Annie, Kindred, Demoncursed Vayne, and Prestige Pandemonium Shaco skins.

---

## Champions Buffed
- Shyvana (AD and AP builds improved for more durability and differentiation)

## Champions Nerfed
- No specific champion nerfs detailed in the available notes.

---

Stay tuned for more in-depth champion changes and gameplay adjustments! Don't forget to check out the new demon-themed skins and try out the revamped runes.


--- https://www.leagueoflegends.com/en-us/news/game-updates/league-of-legends-patch-26-10-notes/ ---


# League of Legends Patch 26.10 Summary

Patch 26.10 continues the Season of Demons with key balance changes to champions, items, and runes. Riot is tuning down some overpowered elements like Deathfire Touch, Gluttonous Greaves, and Voltaic Cyclosword, while buffing Stormraider’s Surge and some new starting items to improve diversity in builds. Lee Sin receives quality of life improvements by removing shield limiters, and Quinn is making a move into the jungle role. Behavior updates include termination votes for disrupted games and better chat detection models. The Essence Emporium event returns May 13 to June 10 with new titles and rewards.

---

## Champions Buffed
- **Ambessa**  
  - Q damage increased on early ranks (target max health damage adjusted)  
  - Reduced bonus monster damage on Q to balance jungle clear  
  - R healing percentage lowered to encourage bruiser playstyle on top lane

- **Quinn**  
  - Jungle role adjustments (details not fully specified in the snippet)

- **Lee Sin**  
  - Shield limiters removed, improving his early trades and survivability  
  - Other modernization and quality of life changes

---

## Champions Nerfed
- No direct nerfs explicitly listed in the provided notes but adjustments to Ambessa's ultimate healing are a minor nerf aspect. Other nerfs are primarily on items and runes.

---

## Item & Rune Changes (Highlights)
- Deathfire Touch: Nerfed  
- Gluttonous Greaves: Nerfed  
- Voltaic Cyclosword: Nerfed  
- Stormraider’s Surge: Buffed  
- New Starting Items: Buffed to encourage adoption

---

## Other Updates
- Termination votes introduced for disrupted games  
- Improved detection for toxic text chat  
- Essence Emporium event with new titles and chromas launching May 13  

---

This patch focuses on balancing high skill counterplay, adjusting jungle vs. top power dynamics, and fostering diverse item/rune strategies. Stay tuned for my gameplay and breakdowns of these changes on stream!


--- https://www.leagueoflegends.com/en-us/news/game-updates/league-of-legends-patch-26-11-notes/ ---


# League of Legends Patch 26.11 - Quick Summary

Patch 26.11 brings a significant focus on rebalancing support champions, especially to curb enchanter dominance and empower melee supports. The meta is shifting toward tankier, more engaging playstyles with systemic tweaks across the board. Top lane sees some changes with Heartsteel buffs and Hexplate nerfs on ranged champs to adjust survivability and scaling. Xin Zhao and Smolder receive nerfs to reduce their sustain and bruiser damage potential. Plus, exciting new skins drop including Italian-themed Sion, Illaoi, Irelia, and Vel'Koz, and stunning Eternal Aspect skins for Leona and Diana. Pride 2026 content and emotes also make their debut!

---

## Champions Buffed
- **Top Lane**: Heartsteel buff (affects champions who build it)
- **Melee Supports**: Various buffs to improve viability (specific champions not detailed)

## Champions Nerfed
- **Ranged Top Laners**: Hexplate nerf to increase health cost
- **Xin Zhao**: Reduced healing
- **Smolder**: Base damage nerfs for bruiser build

---

Stay tuned for detailed champion-specific buffs and nerfs once Riot releases the individual champion patch updates!

And just like that! With this short notebook we have used python and OpenAI's api to generate short scripts for League videos!

## What I keep

This started as a generic website summarizer, but giving a little twist to it and pointing it 
at LoL patch notes immediately made it useful for something I actually do — 
creating short-form content scripts.

A few things worth noting:
- The system prompt does most of the heavy lifting. Adding specifications like "content creator script" completely changes the output's tone and structure.
- The scraper struggles with JavaScript-heavy pages — the LoL patch notes 
  returned partial content, which affected summary quality on some patches.
- Next step that I would take: customize the prompt specifically for patch note structure 
  (buffs, nerfs, new items) to get more consistent output format across patches.